# Trees & Binary Search Trees — Combined Masterclass

The fifth notebook in the DSA series. Trees are the **most recursive** data structure — and the topic where "thinking recursively" finally clicks for most people. Almost every tree problem reduces to one of 6-8 patterns; once you internalize them, the solutions write themselves.

This notebook covers both **general binary trees** (sections 1-8) and **binary search trees** (sections 9-11) because:
1. The traversal mechanics are shared.
2. BST problems often piggyback on tree traversal patterns (especially inorder).
3. Most interviews mix both.

Out of scope: binary search on arrays, sorting algorithms, recursion as a standalone topic, backtracking, dynamic programming on trees.

## How to read this notebook
1. **Section 1** — vocabulary. Trees have lots of subtle terms (height vs depth, complete vs perfect vs balanced) and interviewers test the distinction.
2. **Section 2** — the four traversals. These are non-negotiable. Memorize the recursive forms AND know the iterative versions.
3. **Sections 3-8** — tree patterns. Each opens with a picture/story, then the recursive insight, then code.
4. **Sections 9-11** — BST-specific.
5. **Section 12** — cheat sheet, common bugs, interview narration.

## The 6 recurring tree patterns (memorize these)
1. **Top-down DFS** — pass state DOWN as arguments; check condition at each node (e.g., path sum, validate BST with range)
2. **Bottom-up DFS** — children return info to parent; parent combines (e.g., height, diameter, is-balanced)
3. **BFS / level order** — queue, level-by-level (e.g., right view, min depth, zigzag)
4. **DFS with global state** — recurse and update a `nonlocal` variable (e.g., max path sum, diameter)
5. **LCA pattern** — recursive "if both subtrees return something, I'm the answer"
6. **BST property pattern** — exploit `left < root < right` to prune one entire subtree (effectively binary search on the tree)


---
# 1. Foundations — vocabulary and types

## What is a tree?

A **tree** is a connected, acyclic graph. In data structures, we always work with **rooted trees** — there's a designated root, and every other node has exactly one parent. Every node may have zero or more children.

```
          1            <-- root
         / \
        2   3
       /|   |\
      4 5   6 7        <-- leaves (no children)
```

## Core terminology (interviewers test these)

| Term | Meaning |
|---|---|
| **Root** | The topmost node — no parent |
| **Leaf** | A node with no children |
| **Internal node** | Any non-leaf node |
| **Parent / child** | Direct connections |
| **Ancestor / descendant** | Transitive parent / child relationships |
| **Sibling** | Same parent |
| **Edge** | Connection between parent and child |
| **Depth of a node** | Number of edges from the **root** down to it (root has depth 0) |
| **Height of a node** | Number of edges in the **longest path** from it down to a leaf (leaf has height 0) |
| **Height of the tree** | Height of the root |
| **Level** | All nodes at the same depth |
| **Size** | Total number of nodes |
| **Degree of a node** | Number of children it has |

**The depth-vs-height trap:** these are often confused.
- **Depth** is measured from the root *downward to the node*. Root = depth 0.
- **Height** is measured from the node *down to its deepest leaf*. Leaf = height 0.

You always count edges, not nodes. (Some textbooks use "depth in nodes," which gives root = depth 1. The interview convention is edges-from-root, so root = depth 0.)

## Binary trees and their flavours

A **binary tree** is a tree where each node has at most 2 children, conventionally called `left` and `right`. Most interview questions are on binary trees.

| Flavour | Definition |
|---|---|
| **Full binary tree** | Every node has either 0 or 2 children (no single-child nodes) |
| **Complete binary tree** | All levels fully filled except possibly the last, which fills left to right |
| **Perfect binary tree** | All internal nodes have 2 children AND all leaves at the same depth |
| **Balanced binary tree** | The height difference between left and right subtrees of every node is at most 1 |
| **Skewed binary tree** | Every node has only one child — essentially a linked list |

**Why these distinctions matter:**
- A **complete** tree with N nodes can be stored as a flat array (index 0 = root, children of i at 2i+1 and 2i+2). This is what heaps use.
- A **perfect** tree with height h has exactly `2^(h+1) - 1` nodes.
- A **balanced** tree has height O(log N). Skewed has height O(N) — the difference between a fast BST and a slow one.

## Why trees matter

Trees appear everywhere: file systems, the DOM, JSON, abstract syntax trees, decision trees, indexes (B-trees in databases), heaps (priority queues), expression evaluation. In interviews, they test whether you can think recursively under pressure.


In [ ]:
# 1.1 The Node class — what every binary tree problem starts with
class Node:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    def __repr__(self):
        return f"Node({self.val})"

# Quick builder for testing — takes a level-order list with None for missing nodes
def build_tree(values):
    '''
    Build a binary tree from level-order list.
    Example: [1, 2, 3, None, 4, 5, None] yields:
              1
             / \
            2   3
             \  /
             4 5
    '''
    if not values: return None
    from collections import deque
    root = Node(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        # left child
        if i < len(values) and values[i] is not None:
            node.left = Node(values[i])
            q.append(node.left)
        i += 1
        # right child
        if i < len(values) and values[i] is not None:
            node.right = Node(values[i])
            q.append(node.right)
        i += 1
    return root

# Test
root = build_tree([1, 2, 3, 4, 5, 6, 7])
print(root, root.left, root.right)


---
# 2. The four traversals — foundation of everything

A traversal visits every node exactly once. There are four canonical orders. **Memorize the recursive forms**, then learn the iterative versions for when interviewers ask.

## The picture

```
        1
       / \
      2   3
     / \  / \
    4  5 6  7
```

| Traversal | Order | Visits at this tree |
|---|---|---|
| **Inorder** | Left → Root → Right | 4, 2, 5, 1, 6, 3, 7 |
| **Preorder** | Root → Left → Right | 1, 2, 4, 5, 3, 6, 7 |
| **Postorder** | Left → Right → Root | 4, 5, 2, 6, 7, 3, 1 |
| **Level-order** | Top to bottom, left to right | 1, 2, 3, 4, 5, 6, 7 |

## When to use which

| Traversal | Best for |
|---|---|
| **Inorder** | BSTs — visits nodes in **sorted order**. Used for `kth smallest`, `validate BST`, `find median`. |
| **Preorder** | Copying / serializing a tree (you can rebuild from preorder + inorder). Also "process the node before its children." |
| **Postorder** | Bottom-up computations — height, diameter, deletion. Anything where the parent needs results from BOTH children first. |
| **Level-order (BFS)** | Per-level processing — minimum depth, right view, zigzag, average per level. |

## Complexity (all four)
- **Time:** O(N) — visit every node once
- **Space (recursive):** O(H) for the call stack, where H is the height. O(log N) balanced, O(N) skewed.
- **Space (iterative):** O(H) for the explicit stack (or O(W) for BFS where W is the max width)


In [ ]:
# 2.1 Inorder — recursive
def inorder(root):
    if root is None:
        return []
    return inorder(root.left) + [root.val] + inorder(root.right)

# 2.2 Preorder — recursive
def preorder(root):
    if root is None:
        return []
    return [root.val] + preorder(root.left) + preorder(root.right)

# 2.3 Postorder — recursive
def postorder(root):
    if root is None:
        return []
    return postorder(root.left) + postorder(root.right) + [root.val]

# Build the test tree
root = build_tree([1, 2, 3, 4, 5, 6, 7])
assert inorder(root)  == [4, 2, 5, 1, 6, 3, 7]
assert preorder(root) == [1, 2, 4, 5, 3, 6, 7]
assert postorder(root) == [4, 5, 2, 6, 7, 3, 1]
print("Recursive traversals OK")


## Iterative versions

Interviewers love asking these because they reveal whether you actually understand what recursion is doing. The trick: simulate the call stack with an explicit stack.

### Iterative inorder — push left chain, pop, go right
1. Push the leftmost spine onto the stack.
2. Pop a node, visit it.
3. Move to its right child and repeat.

### Iterative preorder — easy because we visit before recursing
1. Push root.
2. Pop, visit, push right then left (so left pops first).

### Iterative postorder — two tricks

**Easier trick:** do a modified preorder (root → right → left), then reverse. That's postorder.

**Harder trick (true iterative postorder):** use a stack + last-visited pointer. Only push a node's right child after confirming its left subtree is done.

In [ ]:
# 2.4 Iterative inorder
def inorder_iterative(root):
    result = []
    stack = []
    cur = root
    while cur or stack:
        # go to the leftmost — push everything on the way
        while cur:
            stack.append(cur)
            cur = cur.left
        # pop, visit, move right
        cur = stack.pop()
        result.append(cur.val)
        cur = cur.right
    return result

assert inorder_iterative(build_tree([1,2,3,4,5,6,7])) == [4, 2, 5, 1, 6, 3, 7]
print("inorder_iterative OK")


In [ ]:
# 2.5 Iterative preorder
def preorder_iterative(root):
    if not root: return []
    result = []
    stack = [root]
    while stack:
        node = stack.pop()
        result.append(node.val)
        if node.right: stack.append(node.right)      # push right first
        if node.left:  stack.append(node.left)       # so left pops first
    return result

assert preorder_iterative(build_tree([1,2,3,4,5,6,7])) == [1, 2, 4, 5, 3, 6, 7]
print("preorder_iterative OK")


In [ ]:
# 2.6 Iterative postorder — the reverse-of-modified-preorder trick
def postorder_iterative(root):
    if not root: return []
    result = []
    stack = [root]
    while stack:
        node = stack.pop()
        result.append(node.val)                       # collect root
        if node.left:  stack.append(node.left)        # then push LEFT first
        if node.right: stack.append(node.right)       # so RIGHT pops first
    return result[::-1]                               # reverse at the end

assert postorder_iterative(build_tree([1,2,3,4,5,6,7])) == [4, 5, 2, 6, 7, 3, 1]
print("postorder_iterative OK")


In [ ]:
# 2.7 Level-order (BFS) — the workhorse
from collections import deque

def level_order(root):
    if not root: return []
    result = []
    q = deque([root])
    while q:
        node = q.popleft()
        result.append(node.val)
        if node.left:  q.append(node.left)
        if node.right: q.append(node.right)
    return result

assert level_order(build_tree([1,2,3,4,5,6,7])) == [1, 2, 3, 4, 5, 6, 7]
print("level_order OK")


In [ ]:
# 2.8 Level-order — but grouped by level (the form most interviews want)
def level_order_grouped(root):
    if not root: return []
    result = []
    q = deque([root])
    while q:
        level_size = len(q)               # KEY TRICK: snapshot size before processing
        level = []
        for _ in range(level_size):
            node = q.popleft()
            level.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        result.append(level)
    return result

print(level_order_grouped(build_tree([1,2,3,4,5,6,7])))
# [[1], [2, 3], [4, 5, 6, 7]]


**The level-snapshot trick** — `level_size = len(q)` at the start of each level — is the single most reused pattern in BFS-on-tree problems. It works because by the time you've processed `level_size` nodes, every new node in the queue belongs to the next level.

**Practice — traversals**

| Concept | LeetCode |
|---|---|
| Binary tree inorder traversal | LC 94 |
| Binary tree preorder traversal | LC 144 |
| Binary tree postorder traversal | LC 145 |
| Binary tree level order traversal | LC 102 |
| Binary tree zigzag level order traversal | LC 103 |
| Average of levels | LC 637 |
| N-ary tree level order traversal | LC 429 |


---
# 3. Tree property computations — the bottom-up DFS pattern

## The pattern

Almost every "compute a property of the tree" problem follows this recipe:
1. **Recurse on left subtree** → get some info from the left
2. **Recurse on right subtree** → get some info from the right
3. **Combine them at the current node** → return the answer

This is called **bottom-up DFS** or **postorder thinking**. Children report up; parents aggregate.

## When to add global state

Some problems need TWO things at each node:
- A value returned to the parent (e.g., the height)
- A value tracked globally (e.g., the maximum diameter seen anywhere)

For these, declare a `nonlocal` or `self.` variable, update it inside the recursive helper, and have the helper return only what the parent needs.

In [ ]:
# 3.1 Height of a binary tree — the simplest bottom-up DFS
# Definition: height = number of edges in longest root-to-leaf path
# (some problems define it as nodes — read carefully)
def height(root):
    if root is None:
        return -1                          # height of empty tree = -1 in edges convention
                                           # use 0 if "nodes" convention is required
    return 1 + max(height(root.left), height(root.right))

# Most LC problems use the "nodes" convention — depth of leaf = 1, empty = 0
def max_depth(root):
    if root is None:
        return 0
    return 1 + max(max_depth(root.left), max_depth(root.right))

t = build_tree([1, 2, 3, 4, 5, 6, 7])
print("height (edges):", height(t))        # 2
print("max_depth (nodes):", max_depth(t))  # 3


In [ ]:
# 3.2 Size — number of nodes
def size(root):
    if root is None: return 0
    return 1 + size(root.left) + size(root.right)

print(size(build_tree([1, 2, 3, 4, 5, 6, 7])))   # 7


In [ ]:
# 3.3 Diameter of a binary tree — LC 543
# The diameter is the longest path between any two nodes, measured in EDGES.
# It may or may not pass through the root.
#
# KEY INSIGHT: for any node, the longest path *through* that node is
#   (height of left subtree) + (height of right subtree)
# So we compute height during DFS and update a global max along the way.
def diameter(root):
    best = [0]
    def height(node):
        if node is None: return 0
        l = height(node.left)
        r = height(node.right)
        best[0] = max(best[0], l + r)        # update global with path THROUGH this node
        return 1 + max(l, r)                 # return height to parent
    height(root)
    return best[0]

assert diameter(build_tree([1, 2, 3, 4, 5])) == 3
print("diameter OK")


**Why we use a list `best = [0]` instead of `best = 0`:** in Python, inner functions can READ enclosing-scope ints but not REASSIGN them without `nonlocal`. A list is mutable, so `best[0] = ...` works without the `nonlocal` keyword. Both styles are fine; the list is slightly more idiomatic in older code, `nonlocal` is cleaner in modern Python.

In [ ]:
# 3.4 Is the tree height-balanced? LC 110
# A tree is balanced if for EVERY node, abs(height(left) - height(right)) <= 1.
#
# NAIVE: at every node, compute height of left and right -> O(N^2)
# BETTER: combine height computation with the balance check. If any subtree is
#         unbalanced, propagate -1 (sentinel) up. Otherwise return its height.
#
# Time: O(N). Space: O(H).
def is_balanced(root):
    def check(node):
        if node is None: return 0
        l = check(node.left)
        if l == -1: return -1
        r = check(node.right)
        if r == -1: return -1
        if abs(l - r) > 1: return -1
        return 1 + max(l, r)
    return check(root) != -1

assert is_balanced(build_tree([1, 2, 3, 4, 5, 6, 7])) == True
# Skewed tree — not balanced
skewed = Node(1)
skewed.left = Node(2)
skewed.left.left = Node(3)
skewed.left.left.left = Node(4)
assert is_balanced(skewed) == False
print("is_balanced OK")


In [ ]:
# 3.5 Count nodes in a COMPLETE binary tree in O(log^2 N)
# Naive: O(N). Trick: in a perfect subtree, count is 2^h - 1, no recursion needed.
# At each step, compare left-spine height to right-spine height:
#   - equal: it's a perfect tree, return 2^h - 1
#   - different: recurse on both sides (only one of them is non-perfect)
# Each non-perfect subtree contributes one O(log N) recursion + one O(log N) spine check.
def count_nodes_complete(root):
    if root is None: return 0
    # Walk down both spines
    lh = rh = 0
    cur = root
    while cur:
        lh += 1
        cur = cur.left
    cur = root
    while cur:
        rh += 1
        cur = cur.right
    if lh == rh:
        return (1 << lh) - 1       # 2^lh - 1
    return 1 + count_nodes_complete(root.left) + count_nodes_complete(root.right)

print(count_nodes_complete(build_tree([1, 2, 3, 4, 5, 6])))    # 6
print(count_nodes_complete(build_tree([1, 2, 3, 4, 5, 6, 7])))  # 7


**Practice — tree properties**

| Concept | LeetCode |
|---|---|
| Maximum depth of binary tree | LC 104 |
| Minimum depth of binary tree (BFS wins here!) | LC 111 |
| Diameter of binary tree | LC 543 |
| Balanced binary tree | LC 110 |
| Count complete tree nodes | LC 222 |
| Count good nodes in tree | LC 1448 |
| Sum of left leaves | LC 404 |


---
# 4. Views of a binary tree

## The picture

Imagine standing on different sides of a tree. You only see the **first node at each level** from your viewpoint.

```
        1
       / \
      2   3
       \   \
        5   4
```

- **Left view:** [1, 2, 5] (leftmost node at each level)
- **Right view:** [1, 3, 4] (rightmost node at each level)
- **Top view:** what you see looking straight down (first node at each horizontal distance)
- **Bottom view:** what you see looking up (last node at each horizontal distance)

## The trick for all four
**Level-order traversal with a per-level snapshot.** For left view, capture the FIRST node at each level; for right view, the LAST. For top/bottom views, track horizontal distance from the root.

In [ ]:
# 4.1 Right view — LC 199
def right_view(root):
    if not root: return []
    res = []
    q = deque([root])
    while q:
        n = len(q)
        for i in range(n):
            node = q.popleft()
            if i == n - 1:                  # last node of this level
                res.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
    return res

print(right_view(build_tree([1, 2, 3, None, 5, None, 4])))   # [1, 3, 4]


In [ ]:
# 4.2 Left view — symmetric to right view
def left_view(root):
    if not root: return []
    res = []
    q = deque([root])
    while q:
        n = len(q)
        for i in range(n):
            node = q.popleft()
            if i == 0:                      # first node of this level
                res.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
    return res

print(left_view(build_tree([1, 2, 3, None, 5, None, 4])))    # [1, 2, 5]


In [ ]:
# 4.3 Top view — first node seen at each horizontal distance
# Root has hd=0. Left child has hd=parent-1, right child has hd=parent+1.
# Use BFS so we encounter shallower nodes first at each hd.
def top_view(root):
    if not root: return []
    hd_to_val = {}                          # horizontal distance -> first val seen
    q = deque([(root, 0)])
    while q:
        node, hd = q.popleft()
        if hd not in hd_to_val:
            hd_to_val[hd] = node.val
        if node.left:  q.append((node.left, hd - 1))
        if node.right: q.append((node.right, hd + 1))
    return [hd_to_val[k] for k in sorted(hd_to_val)]

print(top_view(build_tree([1, 2, 3, 4, 5, 6, 7])))


In [ ]:
# 4.4 Bottom view — LAST node seen at each horizontal distance
# Same BFS, but always overwrite (we want the most recent / deepest at each hd)
def bottom_view(root):
    if not root: return []
    hd_to_val = {}
    q = deque([(root, 0)])
    while q:
        node, hd = q.popleft()
        hd_to_val[hd] = node.val            # always overwrite
        if node.left:  q.append((node.left, hd - 1))
        if node.right: q.append((node.right, hd + 1))
    return [hd_to_val[k] for k in sorted(hd_to_val)]

print(bottom_view(build_tree([1, 2, 3, 4, 5, 6, 7])))


**Why BFS for top/bottom view, not DFS?** With DFS, you might reach `hd=2` via the left-then-right path FIRST and via the right path SECOND. BFS guarantees shallower nodes are seen first, which matches the visual "looking down from above" intuition.

**Practice — views**

| Concept | LeetCode |
|---|---|
| Right view of binary tree | LC 199 |
| Left view of binary tree | (GFG) |
| Top view of binary tree | (GFG) |
| Bottom view of binary tree | (GFG) |
| Vertical order traversal | LC 314 / LC 987 |
| Boundary of binary tree | LC 545 |


---
# 5. Path problems

## The picture

A "path" in tree problems means **a sequence of connected nodes**. Definitions vary:
- **Root-to-leaf path** — must start at root and end at a leaf
- **Any-to-any path** — any pair of nodes; the path goes up to some LCA and back down (Maximum Path Sum)
- **Downward path** — must go strictly parent → child (no zigzag back up), but not necessarily root or leaf

Always **clarify** with the interviewer which kind of path the problem expects.

## Pattern 1: Top-down DFS for root-to-leaf
Pass the running sum (or path) DOWN through the recursion. Check at leaves.

## Pattern 2: Bottom-up DFS with global max for any-to-any
Each node returns the "best path going one direction" (i.e., into one subtree). The parent considers using BOTH subtrees and updates a global max.

## Pattern 3: Prefix sum on tree for downward paths
Same trick as subarray-sum-equals-K on arrays — store running prefix sums in a dict as you descend, undo when you ascend.

In [ ]:
# 5.1 Path sum — does a root-to-leaf path with given sum exist? LC 112
def has_path_sum(root, target):
    if root is None: return False
    if root.left is None and root.right is None:        # leaf
        return root.val == target
    remaining = target - root.val
    return (has_path_sum(root.left, remaining) or
            has_path_sum(root.right, remaining))

assert has_path_sum(build_tree([5,4,8,11,None,13,4,7,2,None,None,None,1]), 22) == True
print("has_path_sum OK")


In [ ]:
# 5.2 All root-to-leaf paths with given sum — LC 113
def path_sum_paths(root, target):
    res = []
    def dfs(node, remaining, path):
        if node is None: return
        path.append(node.val)
        if node.left is None and node.right is None:
            if node.val == remaining:
                res.append(list(path))                  # COPY — path is reused
        else:
            dfs(node.left,  remaining - node.val, path)
            dfs(node.right, remaining - node.val, path)
        path.pop()                                      # backtrack
    dfs(root, target, [])
    return res

print(path_sum_paths(build_tree([5,4,8,11,None,13,4,7,2,None,None,5,1]), 22))


In [ ]:
# 5.3 Maximum path sum (any to any) — LC 124
# Each node returns the best DOWNWARD path starting at it (into one subtree only).
# We update a global max with the path THROUGH this node (using BOTH subtrees).
#
# The "ignore negative subtrees" trick: take max(child_gain, 0) — if a subtree
# would hurt our sum, we skip it.
def max_path_sum(root):
    best = [float('-inf')]
    def gain(node):
        if node is None: return 0
        l = max(gain(node.left),  0)        # negative? pretend it's not there
        r = max(gain(node.right), 0)
        best[0] = max(best[0], node.val + l + r)        # use both for the through-path
        return node.val + max(l, r)                     # return only one for parent
    gain(root)
    return best[0]

print(max_path_sum(build_tree([-10, 9, 20, None, None, 15, 7])))     # 42 (15→20→7)
print(max_path_sum(build_tree([1, 2, 3])))                            # 6


In [ ]:
# 5.4 Path sum III — count downward paths summing to target — LC 437
# This is "subarray sum equals K" on a tree.
# We DFS top-down, maintain a hashmap of prefix sums seen on the current path.
# At each node, prefix - target tells us how many earlier paths could pair with us.
from collections import defaultdict

def path_sum_count(root, target):
    counts = defaultdict(int)
    counts[0] = 1                           # the empty-prefix base case
    answer = [0]
    def dfs(node, prefix):
        if node is None: return
        prefix += node.val
        answer[0] += counts[prefix - target]
        counts[prefix] += 1
        dfs(node.left, prefix)
        dfs(node.right, prefix)
        counts[prefix] -= 1                 # backtrack — we're leaving this node's subtree
    dfs(root, 0)
    return answer[0]

print(path_sum_count(build_tree([10,5,-3,3,2,None,11,3,-2,None,1]), 8))   # 3


**The backtrack in 5.4 is essential.** When you leave a subtree, the prefix sums you added are no longer "above you" on a current path — they were on a sibling's branch. Without `counts[prefix] -= 1`, you'd double-count paths that cross subtree boundaries (which aren't valid downward paths).

**Practice — paths**

| Concept | LeetCode |
|---|---|
| Path sum | LC 112 |
| Path sum II | LC 113 |
| Path sum III | LC 437 |
| Binary tree maximum path sum | LC 124 |
| Sum root to leaf numbers | LC 129 |
| Binary tree paths | LC 257 |
| Longest univalue path | LC 687 |
| Longest zigzag path | LC 1372 |


---
# 6. Lowest Common Ancestor (LCA)

## The picture

The LCA of two nodes `p` and `q` is **the deepest node that has both `p` and `q` in its subtree** (a node is its own descendant).

```
        3
       / \
      5   1
     / \  / \
    6  2 0  8
       /\
      7  4
```

- LCA(5, 1) = 3
- LCA(5, 4) = 5 (5 is its own ancestor and 4 is in 5's subtree)
- LCA(6, 4) = 5
- LCA(7, 8) = 3

## The recursive insight (THE elegant one)

The recursive solution is one of the cleanest in all of DSA:

```
def lca(root, p, q):
    if root is None or root == p or root == q:
        return root
    left  = lca(root.left, p, q)
    right = lca(root.right, p, q)
    if left and right: return root   # p in one side, q in the other -> we're the answer
    return left or right             # both on same side -> propagate up
```

**Why this works:**
- If `root` is `p` or `q` itself, it's the candidate LCA — return it.
- Recurse on both subtrees.
- If both recursions returned something, `p` and `q` are split across us → we are the LCA.
- If only one returned something, the LCA is below — keep propagating it up.
- If neither, return None.

## Complexity
O(N) time (touch every node once), O(H) space for the recursion stack.

## BST optimization
On a BST, you can use the ordering to descend directly: if both `p` and `q` < current, go left; both > current, go right; otherwise split → current is LCA. O(H) time.

In [ ]:
# 6.1 LCA on a generic binary tree — LC 236
def lca(root, p, q):
    if root is None or root.val == p or root.val == q:
        return root
    left  = lca(root.left,  p, q)
    right = lca(root.right, p, q)
    if left and right:
        return root
    return left if left else right

t = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
print(lca(t, 5, 1).val)       # 3
print(lca(t, 5, 4).val)       # 5
print(lca(t, 6, 4).val)       # 5


In [ ]:
# 6.2 LCA on a BST — exploit the ordering for O(H) without recursion
def lca_bst(root, p, q):
    cur = root
    while cur:
        if p < cur.val and q < cur.val:
            cur = cur.left
        elif p > cur.val and q > cur.val:
            cur = cur.right
        else:
            return cur                  # split point — this is the LCA
    return None

# A BST:        6
#              / \
#             2   8
#            /\  /\
#           0 4 7 9
#             /\
#            3 5
bst = build_tree([6,2,8,0,4,7,9,None,None,3,5])
print(lca_bst(bst, 2, 8).val)   # 6
print(lca_bst(bst, 2, 4).val)   # 2
print(lca_bst(bst, 3, 5).val)   # 4


**Practice — LCA**

| Concept | LeetCode |
|---|---|
| Lowest common ancestor of binary tree | LC 236 |
| Lowest common ancestor of BST | LC 235 |
| LCA when nodes may not exist | LC 1644 |
| LCA with parent pointers | LC 1650 |
| LCA of deepest leaves | LC 1123 |
| Smallest common region | LC 1257 |


---
# 7. Constructing trees from traversal data

## The story

Given one or two traversal arrays, rebuild the tree. The key insight: **a single traversal is not enough** — multiple trees can produce the same preorder OR the same inorder. But **inorder + (preorder or postorder)** uniquely determines a tree (assuming distinct values).

## Why inorder is the key partner
Inorder gives you the LEFT subtree and RIGHT subtree separately: every value left of the root in inorder is in the left subtree, every value right of the root is in the right subtree. The other traversal tells you which value is the root.

## The mechanics for preorder + inorder
1. Preorder's first element is the root.
2. Find that value in inorder. Everything left of it = left subtree's inorder; everything right = right subtree's inorder.
3. The number of left-subtree elements tells you how many preorder elements after the root belong to the left subtree.
4. Recurse.

**Naive:** O(N²) because finding the root's index in inorder is O(N) each time.
**Optimal:** O(N) — precompute a hashmap `inorder_value → index`.

## Building a balanced BST from a sorted array
If the array is already sorted, just take the middle as root, recurse left half / right half. Result is height-balanced. O(N).

In [ ]:
# 7.1 Build tree from preorder + inorder — LC 105
def build_from_preorder_inorder(preorder, inorder):
    index = {v: i for i, v in enumerate(inorder)}    # O(1) root lookup
    pre_idx = [0]
    def build(in_left, in_right):
        if in_left > in_right: return None
        root_val = preorder[pre_idx[0]]
        pre_idx[0] += 1
        root = Node(root_val)
        mid = index[root_val]
        root.left  = build(in_left, mid - 1)         # build LEFT before RIGHT — preorder order!
        root.right = build(mid + 1, in_right)
        return root
    return build(0, len(inorder) - 1)

t = build_from_preorder_inorder([3,9,20,15,7], [9,3,15,20,7])
print(preorder(t), inorder(t))                       # [3,9,20,15,7], [9,3,15,20,7]


In [ ]:
# 7.2 Build tree from inorder + postorder — LC 106
def build_from_inorder_postorder(inorder, postorder):
    index = {v: i for i, v in enumerate(inorder)}
    post_idx = [len(postorder) - 1]
    def build(in_left, in_right):
        if in_left > in_right: return None
        root_val = postorder[post_idx[0]]
        post_idx[0] -= 1
        root = Node(root_val)
        mid = index[root_val]
        root.right = build(mid + 1, in_right)        # build RIGHT before LEFT — postorder is reversed root-first when scanning from end!
        root.left  = build(in_left, mid - 1)
        return root
    return build(0, len(inorder) - 1)

t = build_from_inorder_postorder([9,3,15,20,7], [9,15,7,20,3])
print(preorder(t), inorder(t))


In [ ]:
# 7.3 Sorted array to height-balanced BST — LC 108
def sorted_array_to_bst(arr):
    def build(l, r):
        if l > r: return None
        mid = (l + r) // 2
        root = Node(arr[mid])
        root.left  = build(l, mid - 1)
        root.right = build(mid + 1, r)
        return root
    return build(0, len(arr) - 1)

t = sorted_array_to_bst([-10, -3, 0, 5, 9])
print(inorder(t))         # [-10, -3, 0, 5, 9] — confirms it's a BST


In [ ]:
# 7.4 Serialize and deserialize a binary tree — LC 297
# Preorder traversal with explicit "null" markers — uniquely determines the tree.
def serialize(root):
    out = []
    def go(node):
        if node is None:
            out.append("N")
            return
        out.append(str(node.val))
        go(node.left)
        go(node.right)
    go(root)
    return ",".join(out)

def deserialize(data):
    if not data: return None
    tokens = iter(data.split(","))
    def go():
        v = next(tokens)
        if v == "N":
            return None
        node = Node(int(v))
        node.left  = go()
        node.right = go()
        return node
    return go()

original = build_tree([1, 2, 3, None, None, 4, 5])
s = serialize(original)
print(s)
restored = deserialize(s)
print(level_order(restored))    # [1, 2, 3, 4, 5]


**Practice — construction**

| Concept | LeetCode |
|---|---|
| Construct from preorder + inorder | LC 105 |
| Construct from inorder + postorder | LC 106 |
| Construct from preorder + postorder | LC 889 |
| Sorted array to BST | LC 108 |
| Sorted linked list to BST | LC 109 |
| Serialize and deserialize binary tree | LC 297 |
| Serialize and deserialize BST | LC 449 |
| Construct BST from preorder | LC 1008 |


---
# 8. Tree comparison and modification

A small but important pattern family: "are these two trees the same?" "Is this tree a subtree of another?" "Invert this tree."

The pattern: **dual recursion** — recurse on BOTH trees simultaneously. If both null → match; if one null → mismatch; if values differ → mismatch; otherwise recurse on `(left, left)` and `(right, right)`.

In [ ]:
# 8.1 Same tree — LC 100
def same_tree(p, q):
    if not p and not q: return True
    if not p or not q:  return False
    if p.val != q.val:  return False
    return same_tree(p.left, q.left) and same_tree(p.right, q.right)

print(same_tree(build_tree([1,2,3]), build_tree([1,2,3])))    # True
print(same_tree(build_tree([1,2]),   build_tree([1,None,2]))) # False


In [ ]:
# 8.2 Symmetric tree — LC 101 — mirror check
def is_symmetric(root):
    if root is None: return True
    def mirror(a, b):
        if not a and not b: return True
        if not a or not b:  return False
        if a.val != b.val:  return False
        return mirror(a.left, b.right) and mirror(a.right, b.left)
    return mirror(root.left, root.right)

print(is_symmetric(build_tree([1,2,2,3,4,4,3])))    # True
print(is_symmetric(build_tree([1,2,2,None,3,None,3]))) # False


In [ ]:
# 8.3 Subtree of another tree — LC 572
def is_subtree(root, sub):
    if not sub: return True
    if not root: return False
    return same_tree(root, sub) or is_subtree(root.left, sub) or is_subtree(root.right, sub)

print(is_subtree(build_tree([3,4,5,1,2]), build_tree([4,1,2])))   # True


In [ ]:
# 8.4 Invert binary tree — LC 226 — the famous "Homebrew" question
def invert(root):
    if root is None: return None
    root.left, root.right = invert(root.right), invert(root.left)
    return root

t = invert(build_tree([4,2,7,1,3,6,9]))
print(level_order(t))     # [4, 7, 2, 9, 6, 3, 1]


In [ ]:
# 8.5 Flatten binary tree to linked list (preorder, in place) — LC 114
# After flattening, every node's left = None, and right = next node in preorder.
def flatten(root):
    def helper(node):
        '''Returns the TAIL of the flattened subtree.'''
        if node is None: return None
        left_tail  = helper(node.left)
        right_tail = helper(node.right)
        if left_tail:
            left_tail.right = node.right
            node.right = node.left
            node.left = None
        return right_tail or left_tail or node
    helper(root)

t = build_tree([1, 2, 5, 3, 4, None, 6])
flatten(t)
# Walk via .right to verify
cur = t
order = []
while cur:
    order.append(cur.val)
    cur = cur.right
print(order)              # [1, 2, 3, 4, 5, 6]


**Practice — comparison & modification**

| Concept | LeetCode |
|---|---|
| Same tree | LC 100 |
| Symmetric tree | LC 101 |
| Subtree of another tree | LC 572 |
| Invert binary tree | LC 226 |
| Flatten binary tree to linked list | LC 114 |
| Merge two binary trees | LC 617 |
| Convert binary tree to mirror | (GFG) |


---
# 9. Binary Search Trees — foundations

## The defining property

A **BST** is a binary tree where for **every** node:
- All values in its **left subtree** are **strictly less than** the node's value
- All values in its **right subtree** are **strictly greater**

This is more than just "left child < parent < right child" — it must hold recursively. The classic invalid-BST trick:

```
         5
        / \
       3   8
          / \
         2   9      <-- "2" is on the right of "5", which violates the BST property
                         even though 2 < 8 (its immediate parent)
```

## Why we love BSTs

| Operation | Balanced BST | Skewed BST | Sorted array |
|---|---|---|---|
| Search | O(log N) | O(N) | O(log N) via binary search |
| Insert | O(log N) | O(N) | O(N) |
| Delete | O(log N) | O(N) | O(N) |
| Inorder traversal | O(N) | O(N) | already sorted |
| Predecessor / successor | O(log N) | O(N) | O(1) by index |
| Min / max | O(log N) | O(N) | O(1) |

The trade vs sorted array: arrays beat BSTs at random access but lose at insertion. A **balanced** BST gets you O(log N) for everything.

## The KILLER fact for interviews
**Inorder traversal of a BST visits values in ascending sorted order.** This single property unlocks:
- "Kth smallest in BST" — inorder, stop at k
- "Validate BST" — check that inorder is sorted
- "Two-sum in BST" — inorder + two pointers
- "BST to sorted DLL" — inorder, link as you go

## Why operations are O(log N) on average BUT O(N) worst case
Insert numbers in sorted order into a BST without self-balancing — you get a linked list. **Self-balancing variants** (AVL, Red-Black, Treap) maintain O(log N) by performing rotations on insert / delete. We cover these briefly in section 11.

In [ ]:
# 9.1 Search in BST — O(H) — LC 700
def search_bst(root, key):
    cur = root
    while cur:
        if key == cur.val: return cur
        cur = cur.left if key < cur.val else cur.right
    return None

bst = build_tree([4, 2, 7, 1, 3])
print(search_bst(bst, 2))    # Node(2)
print(search_bst(bst, 5))    # None


In [ ]:
# 9.2 Insert into BST — LC 701
def insert_bst(root, key):
    if root is None: return Node(key)
    if key < root.val:
        root.left  = insert_bst(root.left, key)
    elif key > root.val:
        root.right = insert_bst(root.right, key)
    # equal: skip — assume distinct keys
    return root

bst = None
for v in [4, 2, 7, 1, 3, 5, 6]:
    bst = insert_bst(bst, v)
print(inorder(bst))    # [1, 2, 3, 4, 5, 6, 7]


In [ ]:
# 9.3 Delete from BST — LC 450 — three cases
# Case 1: Node has no children — return None
# Case 2: Node has one child — return that child
# Case 3: Node has two children — find the INORDER SUCCESSOR (smallest in right subtree),
#         copy its value into this node, then delete the successor from right subtree
def delete_bst(root, key):
    if root is None: return None
    if key < root.val:
        root.left = delete_bst(root.left, key)
    elif key > root.val:
        root.right = delete_bst(root.right, key)
    else:
        # found it
        if root.left is None:  return root.right
        if root.right is None: return root.left
        # both children exist — replace with inorder successor
        succ = root.right
        while succ.left:
            succ = succ.left
        root.val = succ.val
        root.right = delete_bst(root.right, succ.val)
    return root

bst = None
for v in [5, 3, 6, 2, 4, 7]:
    bst = insert_bst(bst, v)
bst = delete_bst(bst, 3)
print(inorder(bst))     # [2, 4, 5, 6, 7]


In [ ]:
# 9.4 Min and max in BST — leftmost / rightmost
def bst_min(root):
    cur = root
    while cur and cur.left:
        cur = cur.left
    return cur.val if cur else None

def bst_max(root):
    cur = root
    while cur and cur.right:
        cur = cur.right
    return cur.val if cur else None

bst = None
for v in [4, 2, 7, 1, 3, 5, 6]:
    bst = insert_bst(bst, v)
print(bst_min(bst), bst_max(bst))   # 1 7


In [ ]:
# 9.5 Floor and ceiling in BST
# Floor of x = largest value <= x in the tree
# Ceiling of x = smallest value >= x in the tree
def bst_floor(root, x):
    res = None
    cur = root
    while cur:
        if cur.val == x: return cur.val
        if cur.val < x:
            res = cur.val
            cur = cur.right
        else:
            cur = cur.left
    return res

def bst_ceiling(root, x):
    res = None
    cur = root
    while cur:
        if cur.val == x: return cur.val
        if cur.val > x:
            res = cur.val
            cur = cur.left
        else:
            cur = cur.right
    return res

bst = None
for v in [5, 3, 7, 2, 4, 6, 8]:
    bst = insert_bst(bst, v)
print(bst_floor(bst, 6.5), bst_ceiling(bst, 6.5))    # 6 7
print(bst_floor(bst, 1),   bst_ceiling(bst, 1))      # None 2


---
# 10. BST patterns — the interview hits

These are the BST problems you'll see most often. They almost all reduce to either (a) "use the BST property to prune" or (b) "use inorder traversal because it visits in sorted order."

## Pattern A: Range-based recursion (top-down with bounds)
Track the valid range `(min, max)` allowed at each node. When you recurse left, tighten the upper bound; when you recurse right, tighten the lower. Used for validate-BST and trim-BST.

## Pattern B: Inorder + counter / pointer
Walk in inorder. Stop at the k-th node, or compare to previous, or build a linked list. Used for kth-smallest, validate-BST (alternate), fix-bst, two-sum-bst.

## Pattern C: BST search with pruning
Descend the tree. Skip a whole subtree if its values can't possibly satisfy the query (e.g., the entire left subtree is less than `low` → skip it). Used for range-sum, ceiling, floor.

In [ ]:
# 10.1 Validate BST — LC 98 — Pattern A: range-based recursion
# Each node must lie strictly within (lo, hi). Going left tightens hi; right tightens lo.
def is_valid_bst(root):
    def check(node, lo, hi):
        if node is None: return True
        if node.val <= lo or node.val >= hi:
            return False
        return check(node.left, lo, node.val) and check(node.right, node.val, hi)
    return check(root, float('-inf'), float('inf'))

# The canonical INVALID BST from section 9
bad = Node(5)
bad.left  = Node(3)
bad.right = Node(8)
bad.right.left  = Node(2)              # violates BST property (2 < 5 but in right subtree)
bad.right.right = Node(9)
print(is_valid_bst(bad))               # False

good = Node(5)
good.left  = Node(3)
good.right = Node(8)
good.right.left  = Node(6)
good.right.right = Node(9)
print(is_valid_bst(good))              # True


In [ ]:
# 10.2 Validate BST — Pattern B: inorder traversal, check sortedness
def is_valid_bst_inorder(root):
    prev = [float('-inf')]
    def go(node):
        if node is None: return True
        if not go(node.left): return False
        if node.val <= prev[0]: return False
        prev[0] = node.val
        return go(node.right)
    return go(root)

bst = None
for v in [4, 2, 7, 1, 3, 5, 6]:
    bst = insert_bst(bst, v)
print(is_valid_bst_inorder(bst))       # True


In [ ]:
# 10.3 Kth smallest in BST — LC 230 — inorder with a counter
def kth_smallest(root, k):
    count = [0]
    result = [None]
    def go(node):
        if node is None or result[0] is not None: return
        go(node.left)
        count[0] += 1
        if count[0] == k:
            result[0] = node.val
            return
        go(node.right)
    go(root)
    return result[0]

bst = None
for v in [3, 1, 4, None, 2]:
    if v is not None:
        bst = insert_bst(bst, v)
print(kth_smallest(bst, 1))   # 1
print(kth_smallest(bst, 3))   # 3


In [ ]:
# 10.4 Kth smallest — iterative inorder (cleaner for early-stop)
def kth_smallest_iter(root, k):
    stack = []
    cur = root
    while cur or stack:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        k -= 1
        if k == 0:
            return cur.val
        cur = cur.right
    return -1

bst = None
for v in [5, 3, 6, 2, 4, None, None, 1]:
    if v is not None:
        bst = insert_bst(bst, v)
print(kth_smallest_iter(bst, 3))   # 3


In [ ]:
# 10.5 Inorder successor — smallest value > target
# Walk down: if cur.val > target, this is a candidate; remember it and go left to find smaller.
# Else go right.
def inorder_successor(root, target):
    res = None
    cur = root
    while cur:
        if cur.val > target:
            res = cur.val
            cur = cur.left
        else:
            cur = cur.right
    return res

bst = None
for v in [5, 3, 7, 2, 4, 6, 8]:
    bst = insert_bst(bst, v)
print(inorder_successor(bst, 4))    # 5
print(inorder_successor(bst, 7))    # 8
print(inorder_successor(bst, 8))    # None


In [ ]:
# 10.6 Range sum of BST — LC 938 — Pattern C: BST search with pruning
def range_sum_bst(root, lo, hi):
    if root is None: return 0
    if root.val < lo:                              # entire left subtree + root < lo, skip
        return range_sum_bst(root.right, lo, hi)
    if root.val > hi:                              # entire right subtree + root > hi, skip
        return range_sum_bst(root.left, lo, hi)
    return (root.val
            + range_sum_bst(root.left,  lo, hi)
            + range_sum_bst(root.right, lo, hi))

bst = None
for v in [10, 5, 15, 3, 7, None, 18]:
    if v is not None:
        bst = insert_bst(bst, v)
print(range_sum_bst(bst, 7, 15))   # 32 = 7 + 10 + 15


In [ ]:
# 10.7 Trim BST — LC 669 — return tree with only values in [lo, hi]
def trim_bst(root, lo, hi):
    if root is None: return None
    if root.val < lo:
        return trim_bst(root.right, lo, hi)        # drop root + entire left
    if root.val > hi:
        return trim_bst(root.left,  lo, hi)        # drop root + entire right
    root.left  = trim_bst(root.left,  lo, hi)
    root.right = trim_bst(root.right, lo, hi)
    return root

bst = None
for v in [3, 0, 4, None, 2, None, None, 1]:
    if v is not None:
        bst = insert_bst(bst, v)
trimmed = trim_bst(bst, 1, 3)
print(inorder(trimmed))   # [1, 2, 3]


In [ ]:
# 10.8 Recover BST — LC 99 — two nodes were swapped; fix in place
# Walk inorder. The two out-of-order nodes are the ones to swap.
# Subtlety: if the swapped nodes are adjacent in inorder, you see ONE descent.
# If they're far apart, you see TWO descents. Capture first descent's PREV and
# second descent's CUR (or first descent's CUR if there was no second).
def recover_bst(root):
    first = second = prev = [None]
    def go(node):
        if node is None: return
        go(node.left)
        if prev[0] and prev[0].val > node.val:
            if first[0] is None:
                first[0] = prev[0]
            second[0] = node
        prev[0] = node
        go(node.right)
    go(root)
    first[0].val, second[0].val = second[0].val, first[0].val

# Build a tree where 2 and 4 are swapped
t = Node(4)
t.left  = Node(2); t.right = Node(6)
t.left.left = Node(1); t.left.right = Node(5)   # 5 is misplaced (should be in right side)
t.right.right = Node(7)
recover_bst(t)
print(inorder(t))


In [ ]:
# 10.9 Two sum on BST — LC 653 — inorder gives sorted, then two-pointer
def two_sum_bst(root, target):
    arr = inorder(root)
    l, r = 0, len(arr) - 1
    while l < r:
        s = arr[l] + arr[r]
        if s == target: return True
        if s < target: l += 1
        else: r -= 1
    return False

bst = None
for v in [5, 3, 6, 2, 4, None, 7]:
    if v is not None:
        bst = insert_bst(bst, v)
print(two_sum_bst(bst, 9))    # True (2+7)
print(two_sum_bst(bst, 28))   # False


**Practice — BST patterns**

| Concept | LeetCode |
|---|---|
| Search in BST | LC 700 |
| Insert into BST | LC 701 |
| Delete node in BST | LC 450 |
| Validate BST | LC 98 |
| Kth smallest in BST | LC 230 |
| Inorder successor in BST | LC 285 |
| Range sum of BST | LC 938 |
| Trim a BST | LC 669 |
| Recover BST | LC 99 |
| Two-sum BST | LC 653 |
| Closest BST value | LC 270 |
| Convert sorted array to BST | LC 108 |
| Largest BST in a binary tree | LC 333 (hard, classic) |
| Convert BST to sorted DLL | LC 426 |
| Unique BSTs (Catalan numbers) | LC 96 (DP) |


---
# 11. Self-balancing BSTs — AVL & Red-Black (just enough)

## Why they exist
A plain BST can degrade to O(N) per operation if inserts are adversarial (e.g., always sorted). Self-balancing BSTs perform **rotations** during insert and delete to keep the height O(log N).

You will rarely be asked to implement these in an interview — but the **concepts** (rotations, balance factor, properties) come up in design rounds and conceptual questions.

## AVL trees

- **Invariant:** for every node, `|height(left) - height(right)| <= 1`. This `height(left) - height(right)` is called the **balance factor**.
- **On insert/delete:** check balance factors up the recursion. If any violates, perform a rotation (LL, RR, LR, or RL).
- **Strict balance** → O(log N) height, faster lookups, but more rotations on writes.

### The four rotation cases

| Imbalance | Fix |
|---|---|
| Left-Left (balance > 1, new key in left.left) | Right rotate |
| Right-Right (balance < -1, new key in right.right) | Left rotate |
| Left-Right (balance > 1, new key in left.right) | Left rotate left subtree, then right rotate root |
| Right-Left (balance < -1, new key in right.left) | Right rotate right subtree, then left rotate root |

## Red-Black trees

- **Used by `std::map` in C++, `TreeMap` in Java, and most database indexes' B-tree variants are conceptually similar.**
- **Properties:**
  1. Every node is red or black
  2. Root is black
  3. Every leaf (NIL) is black
  4. Red nodes have black children (no two consecutive reds)
  5. Every path from a node to its descendant leaves has the **same number of black nodes** (called the "black height")

- **Looser than AVL** — height bounded by `2 * log(N+1)`. Fewer rotations on insert/delete, so writes are faster, lookups slightly slower than AVL.

## In Python

Python's standard library doesn't have a built-in balanced BST. Two common workarounds:
- **`sortedcontainers.SortedList` / `SortedDict`** — third-party, very fast, behaves like a balanced BST for most operations
- **`bisect` module** — for sorted lists, supports binary search + insertion (O(N) insert, O(log N) search)

For interview problems where you'd reach for a balanced BST in Java/C++, in Python you often use a different structure (heap + lazy deletion, sorted list, etc.).

In [ ]:
# 11.1 AVL — left and right rotations (the core primitive)
# Right rotation:        Left rotation:
#     z                       z
#    /                         \
#   y                           y
#  /                             \
# x                               x
# After right-rotate(z):    After left-rotate(z):
#     y                          y
#    / \                        / \
#   x   z                      z   x

class AVLNode:
    def __init__(self, val):
        self.val = val
        self.left = self.right = None
        self.height = 1

def avl_height(node):
    return node.height if node else 0

def avl_balance(node):
    return avl_height(node.left) - avl_height(node.right) if node else 0

def right_rotate(z):
    y = z.left
    z.left = y.right
    y.right = z
    z.height = 1 + max(avl_height(z.left), avl_height(z.right))
    y.height = 1 + max(avl_height(y.left), avl_height(y.right))
    return y

def left_rotate(z):
    y = z.right
    z.right = y.left
    y.left = z
    z.height = 1 + max(avl_height(z.left), avl_height(z.right))
    y.height = 1 + max(avl_height(y.left), avl_height(y.right))
    return y

def avl_insert(root, key):
    if root is None: return AVLNode(key)
    if key < root.val:
        root.left = avl_insert(root.left, key)
    elif key > root.val:
        root.right = avl_insert(root.right, key)
    else:
        return root
    root.height = 1 + max(avl_height(root.left), avl_height(root.right))
    balance = avl_balance(root)
    # Left-Left
    if balance > 1 and key < root.left.val:
        return right_rotate(root)
    # Right-Right
    if balance < -1 and key > root.right.val:
        return left_rotate(root)
    # Left-Right
    if balance > 1 and key > root.left.val:
        root.left = left_rotate(root.left)
        return right_rotate(root)
    # Right-Left
    if balance < -1 and key < root.right.val:
        root.right = right_rotate(root.right)
        return left_rotate(root)
    return root

# Insert in sorted order — without rebalancing this would skew
# With AVL, the tree stays balanced
root = None
for v in [1, 2, 3, 4, 5, 6, 7]:
    root = avl_insert(root, v)
# After insertion, root should be 4 (perfectly balanced)
print(root.val, "height:", root.height)    # 4, height: 3


**Practice — self-balancing BSTs (mostly conceptual)**

| Concept | Source |
|---|---|
| AVL insert / delete with rotations | GFG |
| Red-Black tree properties | CLRS chapter 13 |
| Using SortedContainers in Python | LC problems where order + insert matters, e.g., 295 (median from data stream — heap-based alt) |


---
# 12. Pattern-recognition cheat sheet

## 12a. What pattern does this look like?

| Problem signal | Reach for |
|---|---|
| "Compute a property of the tree" (height, diameter, balanced) | Bottom-up DFS — children report up |
| "Pass info from root down" (path sum, depth-tracking) | Top-down DFS — pass state as arguments |
| "Per-level processing" (right view, zigzag, min depth) | BFS with level snapshot |
| "Find a path summing to target" | Top-down DFS + backtracking |
| "Count downward paths summing to target" | Prefix-sum hashmap on tree |
| "Maximum path between any two nodes" | Bottom-up DFS + global max + "ignore negatives" trick |
| "LCA of two nodes" | Recursive: "if both subtrees return something, I'm the answer" |
| "Construct tree from traversals" | inorder + (preorder or postorder); hashmap for O(1) root lookup |
| "Serialize / deserialize" | Preorder with null markers |
| "Validate / compare / mirror tree" | Dual recursion on both trees |
| "Validate BST" | Range-based recursion OR inorder + sortedness check |
| "Kth smallest in BST" | Inorder + counter |
| "Two-sum in BST" | Inorder → sorted array + two pointers |
| "Range sum / trim BST" | BST search with subtree pruning |
| "Floor / ceiling / successor / predecessor in BST" | Descend, capture candidates on the way |
| "Count nodes in complete tree fast" | Compare left-spine vs right-spine heights, use `2^h - 1` shortcut |

## 12b. The 6 master patterns (memorize)

1. **Top-down DFS** — `dfs(node, accumulated_state)`. Pass state down. Used for path-sum, depth tracking, validate-BST.

2. **Bottom-up DFS** — `def f(node): l = f(left); r = f(right); return combine(l, r)`. Used for height, diameter, balanced check.

3. **BFS / level order** — queue + `level_size = len(q)` snapshot. Used for right view, min depth, zigzag.

4. **DFS + global state** — `nonlocal` or `[mutable_holder]`. Returns one thing to parent, updates another globally. Used for diameter, max path sum.

5. **LCA pattern** — `if root in {p, q}: return root; left = ...; right = ...; return root if left and right else (left or right)`.

6. **BST pruning** — exploit `left < root < right` to skip entire subtrees. Used for search, range sum, trim, floor, ceiling.

## 12c. Complexity summary

| Operation | Generic tree | Balanced BST | Skewed BST |
|---|---|---|---|
| Traversal (all 4) | O(N) | O(N) | O(N) |
| Search | O(N) | O(log N) | O(N) |
| Insert / delete | O(N) | O(log N) | O(N) |
| Height computation | O(N) | O(N) | O(N) |
| LCA (generic) | O(N) | O(N) | O(N) |
| LCA (BST-specific) | n/a | O(log N) | O(N) |
| Space (recursive) | O(H) | O(log N) | O(N) |


---
# 13. Interview narration & common bugs

## 13a. The opening narration

Before writing any tree code, narrate:

1. "Is this a generic binary tree, or specifically a BST?"
2. "Are values guaranteed distinct, or can there be duplicates?"
3. "Can it be empty? Can it be a single node?"
4. "Should I expect a balanced tree, or could it be skewed?"
5. "What's the expected input/output format — `TreeNode` objects, level-order list, or something else?"

## 13b. How to explain recursion on trees

> "A binary tree is recursive by definition: a tree is either empty, or it's a root with a left subtree and a right subtree, each of which is itself a tree. So nearly every tree algorithm is a function that handles the empty case, then recursively processes left and right, then combines the results."

## 13c. When to use BFS vs DFS

> "I'd reach for BFS when the answer depends on **levels** — minimum depth, right view, level averages, zigzag — because BFS naturally processes one level at a time. I'd reach for DFS when the answer depends on **paths or aggregated properties** — height, diameter, path sums, validation — because DFS lets me accumulate state along a path or roll up information from leaves to root."

## 13d. Why inorder is special for BSTs

> "Inorder traversal of a BST visits nodes in sorted ascending order. So a lot of BST problems reduce to 'do inorder and apply an array technique to the resulting sorted sequence' — kth smallest, two-sum, validate, check for duplicates, all use this."

## 13e. Common bugs

1. **Forgetting the `if root is None` base case.** Every recursive tree function needs it as the first line.

2. **Returning value vs returning Node.** Some helpers return the node, others return an integer like height. Be consistent inside a single function.

3. **Inorder for BST but not checking strictness.** If the problem says "strictly increasing" (distinct values), `<=` is a bug — use `<`.

4. **Using `<` vs `<=` in the range check for `is_valid_bst`.** Strict if values must be distinct (`if node.val <= lo or node.val >= hi: return False`). The classic bug is `<` and `>` which allows equal values.

5. **Path-sum problems that don't backtrack.** When you append to `path` going down, you MUST `path.pop()` going up. Otherwise sibling subtrees see polluted state.

6. **Forgetting that "leaf" means BOTH children are None.** A node with one child is not a leaf.

7. **Building tree from preorder + inorder, processing children in wrong order.** Build LEFT first when using preorder (because preorder visits root → left → right). Build RIGHT first when scanning postorder backwards.

8. **Iterative inorder forgetting to push the left chain first.** The pattern is: while cur, push and go left; pop; visit; go right.

9. **In LCA, returning the wrong thing when only one subtree finds a target.** The correct return is `left or right` — propagate whichever one isn't None.

10. **Max path sum forgetting to ignore negative subtrees.** `max(gain(child), 0)` is the trick; without it, including a heavily negative subtree drags your answer down.

11. **Diameter computed in nodes vs edges.** Edges = nodes - 1. LeetCode uses edges; some interview problems use nodes. Clarify.

12. **Recursion depth limit on deep trees.** Python defaults to 1000. For unbalanced trees with N=10^4 nodes, `sys.setrecursionlimit(10**6)` or use iterative versions.

13. **BFS using `list.pop(0)` instead of `deque.popleft()`.** O(N) per pop → makes the algorithm O(N²). Always import `deque`.

14. **In BST delete, replacing with successor's value but not deleting the successor.** The two-child case needs BOTH: copy successor's value AND recursively delete the successor from the right subtree.

15. **For top/bottom view, assuming horizontal distance is an integer 0..N.** It can go negative for deep left chains. Don't index into an array — use a dict and sort keys at the end.
